In [3]:
from torchvision.datasets import GTSRB
from torchvision import transforms 
import torch
import torch.nn as nn
from torch.optim import Adam
import  torchvision.transforms.v2 as transforms
import torchvision.transforms.functional as F 
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [4]:
from pathlib import Path

print(Path.cwd())
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

DATA_DIR.mkdir(exist_ok=True)


C:\Users\user\deep-learning-traffic-signs\notebooks


In [5]:
basic_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=.2 , contrast=0.5),
    transforms.RandomResizedCrop((64,64),scale=(0.9,1),ratio=(1,1)),
])
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])
train_dataset = GTSRB(
    root=str(DATA_DIR),
    split="train",
    transform=basic_transform,
    download=True
)

test_dataset = GTSRB(
    root=str(DATA_DIR),
    split="test",
    transform=test_transform,
    download=True
)
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
 
)
train_N=len(train_loader.dataset)
test_N=len(test_loader.dataset)

C:\Users\user\deep-learning-traffic-signs\.venv\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [ ]:
class MyConvBlock( nn.Module ):
    def __init__(self,in_ch,out_ch,dropout_p):
        kernel_size=3
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,kernel_size,stride=1,padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.MaxPool2d(2,stride=2)
        )
    def forward(self,x):
        return self.model(x)
flattened_img_size=128*8*8
N_CLASSES=43
IMG_CHS=3
IMG_WIDH=64
IMG_LENGHT=64
base_model= nn.Sequential(
    MyConvBlock(IMG_CHS,32,0), #(32,32,32)
    MyConvBlock(32,64,0.2),#(64,16,16)
    MyConvBlock(64,128,0),#(128,8,8)
    nn.Flatten(),
    nn.Linear(flattened_img_size,1024),
    nn.Dropout(.3),
    nn.ReLU(),
    nn.Linear(1024,N_CLASSES)
)
loss_function=nn.CrossEntropyLoss()
optimizer=Adam(base_model.parameters())
torch._dynamo.config.suppress_errors = True
device=torch.device("cpu")
model=torch.compile(base_model.to(device))
model

In [ ]:
def get_batch_accuracy(output,y,N):
    pred=output.argmax(dim=1,keepdim=True)
    correct=pred.eq(y.view_as(pred)).sum().item()
    return correct/N


In [ ]:
def train():
    loss=0
    accuracy=0
    model.train()
    for x,y in train_loader:
        output=model(x)
        optimizer.zero_grad()
        batch_loss=loss_function(output,y)
        batch_loss.backward()
        optimizer.step()
        loss += batch_loss.item()
        accuracy += get_batch_accuracy(output,y,train_N)
    print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))
    

In [ ]:
def validate():
    loss=0
    accuracy=0
    model.eval()
    with torch.no_grad():
        for x,y in test_loader:
            output=model(x)
            loss+=loss_function(output,y).item()
            accuracy += get_batch_accuracy(output,y,test_N)
        print('Valid-Loss:{:.4f} Accuracy {:.4f}'.format(loss,accuracy))

In [ ]:
epochs=15
for epoch in range (epochs):
    print ('epoch:{}'.format(epoch))   
    train()
    validate()